# 00 — Demo del ETL, paso a paso

Este notebook muestra el flujo de `src/etl.py` **una etapa por celda**, con el resultado de cada una.
No tiene lógica propia: llama a las mismas funciones del script.

La forma oficial de correr todo sigue siendo un solo comando:

```bash
python src/etl.py
```

```
datasets/*.csv  →  raw_ (copia fiel)  →  core_ (limpio)  →  mart_ (para analizar)  →  export/*.csv
```

In [11]:
import sys
import pandas as pd

sys.path.append("../src")
import etl
from db import conectar

def ver(sql):
    with conectar() as conn:
        return pd.read_sql(sql, conn)

## 1. Extraer: los CSV entran a `raw_` tal como vienen

Se borra la base y se cargan los dos CSV **sin corregir nada**. Al terminar, se controla que la
base tenga las mismas filas que los archivos.

In [12]:
etl.extraer()


=== 1. EXTRAER (datasets/*.csv -> raw_)
Ejecutando 01_raw_tables.sql ...
Cargando freMTPL2freq.csv -> raw_freq ...
Cargando freMTPL2sev.csv -> raw_sev ...

Control de carga
tabla        filas CSV    filas DB  estado
raw_freq       678,013     678,013  OK
raw_sev         26,639      26,639  OK
    [OK] filas freq: 678,013 -> 678,013
    [OK] filas sev: 26,639 -> 26,639


En `raw_` los errores siguen ahí. Por ejemplo, una póliza con el ID escrito como `1.00E+05`
(formato Excel), que obliga a guardar la columna como texto:

In [13]:
ver("""
    SELECT idpol, typeof(idpol) AS tipo, claimnb, exposure, region
    FROM raw_freq
    WHERE idpol GLOB '*E*'
""")

,idpol,tipo,claimnb,exposure,region
0,1.00E+05,text,0,0.92,Rhone-Alpes


## 2. Limpiar: de `raw_` a `core_`

`sql/20_limpieza_core.sql` aplica las decisiones de calidad (`docs/05_limpieza.md`):
el conteo de siniestros sale de la tabla de montos, los siniestros sin póliza se apartan,
la exposición se recorta a 1 año y las regiones se normalizan. Nada se borra: lo dudoso se **marca**.

In [ ]:
etl.limpiar()


=== 2. LIMPIAR (raw_ -> core_)
Ejecutando 02_core_tables.sql ...
Ejecutando 20_limpieza_core.sql ...

Control de carga
  pólizas:    raw_freq 678,013 -> core_poliza 678,013  OK
  siniestros: raw_sev 26,639 -> core_siniestro 26,444 + huérfanos 195 = 26,639  OK
  claves foráneas rotas: 0  OK
    [OK] pólizas: 678,013 -> 678,013
    [OK] siniestros (con póliza + huérfanos): 26,639 -> 26,639
    [OK] pólizas con conteo inconsistente: 0 -> 0
    [OK] claves foráneas rotas: 0 -> 0


Pólizas que declaraban siniestros sin monto: el valor original queda en `claimnb_declarado`.

In [15]:
ver("""
    SELECT idpol, claimnb_declarado, claimnb, flag_claimnb_corregido, flag_lote_idpol_bajo
    FROM core_poliza
    WHERE flag_claimnb_corregido = 1
    LIMIT 5
""")

,idpol,claimnb_declarado,claimnb,flag_claimnb_corregido,flag_lote_idpol_bajo
0,1,1,0,1,1
1,3,1,0,1,1
2,5,1,0,1,1
3,10,1,0,1,1
4,11,1,0,1,1


El efecto en el indicador principal, la frecuencia (siniestros por año de exposición):

In [16]:
ver("""
    SELECT 'antes (raw)' AS momento, ROUND(SUM(claimnb) / SUM(exposure), 4) AS frecuencia FROM raw_freq
    UNION ALL
    SELECT 'después (core)', ROUND(SUM(claimnb) / SUM(exposure), 4) FROM core_poliza
""")

,momento,frecuencia
0,antes (raw),0.1007
1,después (core),0.0738


## 3. Modelar: de `core_` a `mart_` y a `export/`

`sql/30_mart.sql` arma una tabla plana con **una fila por póliza**: sus siniestros y montos ya sumados
(primero se suma y después se une, para no duplicar filas) y las franjas de edad, BonusMalus y
antigüedad. Los siniestros grandes se recortan al percentil 99.
`sql/31_mart_powerbi.sql` arma las tablas chicas del dashboard.

In [17]:
etl.modelar(percentil=0.99)


=== 3. MODELAR (core_ -> mart_ -> export/), tope en el percentil 0.99
Tope: percentil 0.99 = 16,451.22. 265 siniestros lo superan; el tope recorta el 30.7 % del monto.
  export/polizas.csv: 678,013 filas
  export/siniestros.csv: 26,444 filas
  export/parametros.csv: 2 filas
  export/calidad_hallazgos.csv: 13 filas
  export/calidad_antes_despues.csv: 3 filas
  export/pareto.csv: 101 filas
  export/dim_region.csv: 13 filas
    [OK] pólizas: 678,013 -> 678,013
    [OK] siniestros: 26,444 -> 26,444
    [OK] monto total en centavos: 5,990,921,650 -> 5,990,921,650


El dataset final que usan Power BI y el notebook de EDA. Acá, pólizas con algún siniestro grande, para ver el tope en acción:

In [18]:
polizas = pd.read_csv("../export/polizas.csv")
polizas[polizas["siniestros_grandes"] > 0][["idpol", "exposure", "siniestros", "monto", "monto_topeado",
         "franja_edad", "franja_bm", "franja_vehage", "area", "region_2016"]].head()

,idpol,exposure,siniestros,monto,monto_topeado,franja_edad,franja_bm,franja_vehage,area,region_2016
2553,5319,0.42,2,32186.45,17004.864,41-50,050,00,D,Île-de-France
5618,17711,0.49,1,69348.00,16451.224,41-50,050,03-05,A,Bretagne
5858,18179,0.72,1,25074.81,16451.224,71+,050,11-15,A,Centre-Val de Loire
6255,18916,0.33,1,66458.90,16451.224,41-50,050,06-10,B,Centre-Val de Loire
6754,19883,1.00,1,30829.41,16451.224,41-50,051-075,01-02,A,Centre-Val de Loire


In [19]:
ver("SELECT * FROM mart_parametro")

,nombre,valor
0,tope_percentil,0.990
1,tope_siniestro,16451.224


## Resultado

Si algún control hubiera fallado, la etapa se habría detenido con `ControlFallido` y no se exportaría
nada. Los archivos que quedan para el equipo:

In [20]:
pd.DataFrame(
    [(f.name, round(f.stat().st_size / 1e6, 2)) for f in sorted(etl.EXPORT.glob("*.csv"))],
    columns=["archivo", "MB"],
)

,archivo,MB
0,calidad_antes_despues.csv,0.00
1,calidad_hallazgos.csv,0.00
2,dim_region.csv,0.00
3,parametros.csv,0.00
4,pareto.csv,0.00
5,polizas.csv,74.52
6,siniestros.csv,0.79
